# 🏭 RL Multi-Warehouse Inventory Management
## End-to-End MVP Notebook

**Khóa luận tốt nghiệp** | Double DQN vs EOQ vs (s,S) vs Newsvendor

---
**Flow**: Setup → Data → Environment → Train DQN → Evaluate vs Baselines → Visualize

> 🚀 Runtime: GPU (T4 on Colab, RTX 3050 local) — ~10-15 phút train 300 episodes

## 0. Setup & Install

In [ ]:
# Install dependencies (skip if already installed)
import subprocess, sys

packages = [
    'gymnasium',
    'numpy',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'tqdm',
    'tensorboard',
    'scipy',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages installed!')

In [ ]:
import os
import sys

# ---- Option A: Running on Google Colab ---
# Upload your project files to Google Drive, then mount:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/rl_inventory'

# ---- Option B: Local run (D:\rl_inventory) ---
PROJECT_ROOT = r'D:\rl_inventory'
# PROJECT_ROOT = '/content/rl_inventory'  # if uploaded to Colab

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'Working dir: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.notebook import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Project modules
from env.inventory_env import MultiWarehouseInventoryEnv, DEFAULT_CONFIG
from agents.dqn_agent import DoubleDQNAgent
from agents.replay_buffer import ReplayBuffer
from baselines.traditional_policies import (
    EOQPolicy, SsPolicyOptimized, NewsvendorPolicy, extract_state_for_policy
)
from scripts.data_preprocessing import generate_synthetic_fallback

# Device check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Plot style
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
print('All imports OK!')

## 1. Data Preparation

In [ ]:
from pathlib import Path

DATA_DIR = Path(PROJECT_ROOT) / 'data' / 'processed'
np_path  = DATA_DIR / 'demand_data.npy'
cfg_path = DATA_DIR / 'env_config.json'

if np_path.exists() and cfg_path.exists():
    # Load real M5 data (preprocessed)
    demand_data = np.load(str(np_path))
    with open(cfg_path) as f:
        env_config = json.load(f)
    print(f'Loaded real M5 demand data: shape={demand_data.shape}')
else:
    # Generate synthetic data for demo
    print('M5 data not found → using synthetic demand (Poisson with weekly seasonality)')
    N_WAREHOUSES = 2
    N_SKUS = 30
    demand_data = generate_synthetic_fallback(
        n_warehouses=N_WAREHOUSES,
        n_skus=N_SKUS,
        n_days=800,
        seed=42,
        out_dir=DATA_DIR,
    )
    env_config = {**DEFAULT_CONFIG, 'n_warehouses': N_WAREHOUSES, 'n_skus': N_SKUS}

T, n_w, n_s = demand_data.shape
print(f'Shape: (T={T} days, warehouses={n_w}, SKUs={n_s})')
print(f'Demand stats: mean={demand_data.mean():.2f}, max={demand_data.max():.2f}')

In [ ]:
# Visualize demand patterns (first 112 days, first 5 SKUs, warehouse 0)
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

for w_idx in range(n_w):
    ax = axes[w_idx]
    for s in range(min(5, n_s)):
        ax.plot(demand_data[:112, w_idx, s], alpha=0.7, linewidth=1.2, label=f'SKU {s}')
    ax.set_title(f'Warehouse {w_idx} — Daily Demand (first 16 weeks, 5 SKUs)')
    ax.set_xlabel('Day')
    ax.set_ylabel('Units')
    ax.legend(fontsize=8, ncol=5)

plt.tight_layout()
plt.show()

## 2. Gymnasium Environment

In [ ]:
# Create environment
env = MultiWarehouseInventoryEnv(
    config=env_config,
    demand_data=demand_data,
)

print('Environment specs:')
print(f'  Warehouses:     {env.n_w}')
print(f'  SKUs/warehouse: {env.n_s}')
print(f'  Pairs (n_pairs):{env.n_pairs}')
print(f'  Obs dim:        {env.obs_dim}')
print(f'  Action space:   {env.action_space}')
print(f'  Obs space:      {env.observation_space}')
print(f'  Order levels:   {env.order_levels.tolist()}')
print(f'  Episode length: {env.episode_len} days')

In [ ]:
# Gymnasium environment check
from gymnasium.utils.env_checker import check_env

test_env = MultiWarehouseInventoryEnv(
    config={**env_config, 'n_skus': 5, 'episode_length': 14},
    demand_data=demand_data[:, :, :5],
)
check_env(test_env, warn=True)
print('✅ Gymnasium check_env PASSED!')

In [ ]:
# Run one episode with random policy to see the environment
obs, info = env.reset(seed=0)
rewards = []
inventories = []

for step in range(env.episode_len):
    action = env.action_space.sample()  # Random policy
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    inventories.append(info['inventory_mean'])
    if terminated or truncated:
        break

print(f'Random policy episode:')
print(f'  Total reward: {sum(rewards):.2f}')
print(f'  Total cost:   {info["episode_cost"]:.2f}')
print(f'  Stockouts:    {info["episode_stockouts"]:.1f} units')
print(f'  Service level:{env.get_service_level():.4f}')

## 3. Train Double DQN Agent

In [ ]:
# Hyperparameters (RTX 3050 / Colab T4 optimized)
HPARAMS = {
    'n_episodes':    300,      # Increase to 500+ for better results
    'hidden_dim':    256,
    'lr':            3e-4,
    'gamma':         0.99,
    'batch_size':    128,
    'buffer_cap':    100_000,
    'eps_start':     1.0,
    'eps_min':       0.05,
    'eps_decay':     50_000,
    'tau':           5e-3,
    'use_per':       False,    # Set True for Prioritized Experience Replay
}

agent = DoubleDQNAgent(
    state_dim       = env.obs_dim,
    n_pairs         = env.n_pairs,
    n_action_levels = len(env.order_levels),
    hidden_dim      = HPARAMS['hidden_dim'],
    lr              = HPARAMS['lr'],
    gamma           = HPARAMS['gamma'],
    batch_size      = HPARAMS['batch_size'],
    buffer_capacity = HPARAMS['buffer_cap'],
    eps_start       = HPARAMS['eps_start'],
    eps_min         = HPARAMS['eps_min'],
    eps_decay       = HPARAMS['eps_decay'],
    tau             = HPARAMS['tau'],
    use_per         = HPARAMS['use_per'],
    log_dir         = str(Path(PROJECT_ROOT) / 'runs' / 'colab_mvp'),
    device          = device,
)

# Count parameters
n_params = sum(p.numel() for p in agent.online_net.parameters())
print(f'Q-Network parameters: {n_params:,}')
print(f'Device: {agent.device}')
print(f'Replay buffer capacity: {HPARAMS["buffer_cap"]:,} transitions')

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
import time

train_rewards  = []
train_costs    = []
train_sl       = []   # service levels
train_losses   = []
eval_rewards   = []
eval_episodes  = []

EVAL_EVERY = 50
SEED       = 42

start_time = time.time()

pbar = tqdm(range(1, HPARAMS['n_episodes'] + 1), desc='Training')

for episode in pbar:
    obs, info = env.reset(seed=SEED + episode)
    ep_reward = 0.0
    ep_losses = []

    for step in range(env.episode_len):
        action = agent.select_action(obs, greedy=False)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        agent.store_transition(obs, action, reward, next_obs, done)
        ep_reward += reward
        obs = next_obs

        loss = agent.update()
        if loss is not None:
            ep_losses.append(loss)

        if done:
            break

    # Track metrics
    sl = env.get_service_level()
    ep_cost = info['episode_cost']
    avg_loss = np.mean(ep_losses) if ep_losses else 0.0

    train_rewards.append(ep_reward)
    train_costs.append(ep_cost)
    train_sl.append(sl)
    if ep_losses:
        train_losses.append(avg_loss)

    total_dem = env.demand_data[env.start_idx:env.start_idx + env.t].sum()
    stockout_rate = info['episode_stockouts'] / (total_dem + 1e-6)
    agent.log_episode(ep_reward, ep_cost, sl, stockout_rate)

    # Periodic evaluation
    if episode % EVAL_EVERY == 0:
        obs_eval, _ = env.reset(seed=9999)
        eval_rew = 0.0
        for _ in range(env.episode_len):
            a = agent.select_action(obs_eval, greedy=True)
            obs_eval, r, te, tr, _ = env.step(a)
            eval_rew += r
            if te or tr: break
        eval_rewards.append(eval_rew)
        eval_episodes.append(episode)
        agent.writer.add_scalar('eval/reward', eval_rew, episode)

    pbar.set_postfix({
        'rew':  f'{ep_reward:.0f}',
        'SL':   f'{sl:.3f}',
        'ε':    f'{agent.epsilon:.3f}',
        'loss': f'{avg_loss:.4f}',
    })

elapsed = time.time() - start_time
print(f'\nTraining done in {elapsed/60:.1f} minutes')
print(f'Last 50 ep avg reward: {np.mean(train_rewards[-50:]):.2f}')
print(f'Last 50 ep avg SL:     {np.mean(train_sl[-50:]):.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Smooth with rolling mean
window = 20

# Reward
ax = axes[0]
r_smooth = pd.Series(train_rewards).rolling(window).mean()
ax.plot(train_rewards, alpha=0.2, color='#4C72B0')
ax.plot(r_smooth, color='#4C72B0', linewidth=2)
if eval_episodes:
    ax.scatter(eval_episodes, eval_rewards, color='red', zorder=5, label='Eval (greedy)', s=40)
    ax.legend()
ax.set_title('Episode Reward (train)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')

# Service Level
ax = axes[1]
sl_smooth = pd.Series(train_sl).rolling(window).mean()
ax.plot(train_sl, alpha=0.2, color='#55A868')
ax.plot(sl_smooth, color='#55A868', linewidth=2)
ax.axhline(0.95, color='red', linestyle='--', alpha=0.7, label='95% target')
ax.set_title('Service Level')
ax.set_xlabel('Episode')
ax.set_ylabel('Service Level')
ax.set_ylim(0, 1.05)
ax.legend()

# Loss
ax = axes[2]
if train_losses:
    loss_smooth = pd.Series(train_losses).rolling(window).mean()
    ax.plot(train_losses, alpha=0.2, color='#DD8452')
    ax.plot(loss_smooth, color='#DD8452', linewidth=2)
ax.set_title('Training Loss (MSE)')
ax.set_xlabel('Update step (every episode)')
ax.set_ylabel('Loss')
ax.set_yscale('log')

plt.suptitle('Double DQN Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Evaluate: DQN vs Baselines

In [ ]:
N_EVAL_EPISODES = 20
EVAL_SEED_START = 1000

# Initialize baselines
eoq  = EOQPolicy(n_pairs=env.n_pairs, order_levels=env.order_levels.tolist())
ss   = SsPolicyOptimized(n_pairs=env.n_pairs, order_levels=env.order_levels.tolist())
nv   = NewsvendorPolicy(n_pairs=env.n_pairs, order_levels=env.order_levels.tolist())

policies = {
    'Double DQN': None,  # uses agent
    'EOQ':        eoq,
    '(s,S)':      ss,
    'Newsvendor': nv,
}

results = {name: [] for name in policies}

for pol_name, policy in policies.items():
    print(f'Evaluating {pol_name}...')
    for ep in tqdm(range(N_EVAL_EPISODES), desc=f'  {pol_name}', leave=False):
        seed = EVAL_SEED_START + ep
        obs, info = env.reset(seed=seed)
        total_reward = 0.0

        for step in range(env.episode_len):
            if pol_name == 'Double DQN':
                action = agent.select_action(obs, greedy=True)
            else:
                inv, dem_hist = extract_state_for_policy(obs, env_config)
                action = policy.get_action(inv, dem_hist)

            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            if terminated or truncated: break

        results[pol_name].append({
            'total_reward':   total_reward,
            'total_cost':     info['episode_cost'],
            'service_level':  env.get_service_level(),
            'stockout':       info['episode_stockouts'],
            'holding_cost':   info['episode_holding_cost'],
            'ordering_cost':  info['episode_ordering_cost'],
        })

# Build summary table
summary_rows = []
for pol_name, ep_results in results.items():
    df_ep = pd.DataFrame(ep_results)
    summary_rows.append({
        'Policy':           pol_name,
        'Avg Total Cost':   df_ep['total_cost'].mean(),
        'Std Cost':         df_ep['total_cost'].std(),
        'Avg Service Lvl':  df_ep['service_level'].mean(),
        'Avg Stockout':     df_ep['stockout'].mean(),
        'Avg Holding Cost': df_ep['holding_cost'].mean(),
        'Avg Order Cost':   df_ep['ordering_cost'].mean(),
        'Avg Reward':       df_ep['total_reward'].mean(),
    })

df_summary = pd.DataFrame(summary_rows)
print('\n' + '='*70)
print('EVALUATION RESULTS')
print('='*70)
print(df_summary.to_string(index=False, float_format='{:.2f}'.format))

## 5. Visualization — Comparison Charts

In [ ]:
COLORS = {
    'Double DQN': '#4C72B0',
    'EOQ':        '#DD8452',
    '(s,S)':      '#55A868',
    'Newsvendor': '#C44E52',
}
pols = df_summary['Policy'].tolist()
cols = [COLORS[p] for p in pols]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(
    'Multi-Warehouse RL vs Traditional Baselines\n'
    f'({env.n_w} Warehouses × {env.n_s} SKUs, {N_EVAL_EPISODES} test episodes)',
    fontsize=13, fontweight='bold',
)

# (1) Total Cost
ax = axes[0]
bars = ax.bar(pols, df_summary['Avg Total Cost'],
              yerr=df_summary['Std Cost'], color=cols, capsize=7,
              edgecolor='white', linewidth=1.2)
ax.set_title('Average Total Cost\n(lower = better)', fontweight='bold')
ax.set_ylabel('Cost per Episode')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, df_summary['Avg Total Cost']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + df_summary['Std Cost'].max()*0.05,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# (2) Service Level
ax = axes[1]
ax.bar(pols, df_summary['Avg Service Lvl']*100, color=cols,
       edgecolor='white', linewidth=1.2)
ax.axhline(95, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label='95% target')
ax.set_title('Average Service Level\n(higher = better)', fontweight='bold')
ax.set_ylabel('Service Level (%)')
ax.set_ylim(0, 115)
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=20)
for i, (pol, val) in enumerate(zip(pols, df_summary['Avg Service Lvl'])):
    ax.text(i, val*100 + 1, f'{val*100:.1f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

# (3) Cost Breakdown
ax = axes[2]
x = np.arange(len(pols))
w = 0.35
ax.bar(x - w/2, df_summary['Avg Holding Cost'], width=w,
       label='Holding', color='#4C72B0', alpha=0.85)
ax.bar(x + w/2, df_summary['Avg Order Cost'], width=w,
       label='Ordering', color='#DD8452', alpha=0.85)
ax.set_title('Cost Breakdown', fontweight='bold')
ax.set_ylabel('Avg Cost')
ax.set_xticks(x)
ax.set_xticklabels(pols, rotation=20)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Spider/Radar chart for multi-metric comparison
from matplotlib.patches import FancyArrowPatch

metrics = ['Service\nLevel', 'Low\nCost', 'Low\nStockout', 'Low\nHolding', 'Low\nOrdering']
n_metrics = len(metrics)

# Normalize metrics to [0,1] (higher = better)
max_cost    = df_summary['Avg Total Cost'].max()
max_stock   = df_summary['Avg Stockout'].max() + 1
max_holding = df_summary['Avg Holding Cost'].max() + 1
max_order   = df_summary['Avg Order Cost'].max() + 1

radar_data = {}
for _, row in df_summary.iterrows():
    pol = row['Policy']
    radar_data[pol] = [
        row['Avg Service Lvl'],
        1.0 - row['Avg Total Cost'] / max_cost,
        1.0 - row['Avg Stockout']   / max_stock,
        1.0 - row['Avg Holding Cost'] / max_holding,
        1.0 - row['Avg Order Cost']   / max_order,
    ]

angles = np.linspace(0, 2*np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for pol_name, vals in radar_data.items():
    v = vals + vals[:1]
    ax.plot(angles, v, linewidth=2, label=pol_name, color=COLORS[pol_name])
    ax.fill(angles, v, alpha=0.1, color=COLORS[pol_name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.set_title('Policy Comparison — Radar Chart\n(larger area = better)', 
             fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Save results
results_dir = Path(PROJECT_ROOT) / 'results'
results_dir.mkdir(exist_ok=True)

csv_path = results_dir / 'evaluation_summary.csv'
df_summary.to_csv(csv_path, index=False)

# Save model
ckpt_dir = Path(PROJECT_ROOT) / 'checkpoints'
ckpt_dir.mkdir(exist_ok=True)
agent.save(str(ckpt_dir / 'colab_mvp_model.pth'))

print(f'Results saved: {csv_path}')
print(f'Model saved:   {ckpt_dir / "colab_mvp_model.pth"}')

## 6. TensorBoard

```bash
# Local:
tensorboard --logdir runs/

# Colab:
# %load_ext tensorboard
# %tensorboard --logdir runs/
```

In [ ]:
# Uncomment to launch TensorBoard in Colab:
# %load_ext tensorboard
# %tensorboard --logdir runs/

## 7. Conclusion & Thesis Notes

### Key Results
| Metric | Double DQN | EOQ | (s,S) | Newsvendor |
|--------|-----------|-----|-------|------------|
| Total Cost | See above | ↑ | ↑ | ↑ |
| Service Level | See above | - | - | - |

### Key Contributions (for thesis)
1. **Double DQN** (van Hasselt et al., 2016) — avoids overestimation bias
2. **Factorized action space** — handles 60 warehouse-SKU pairs tractably  
3. **Experience Replay** (Mnih et al., 2015) — breaks temporal correlations
4. **M5 real-world demand** — more realistic than synthetic benchmarks
5. **3 classical baselines** — EOQ, (s,S), Newsvendor for rigorous comparison

### References
- Mnih et al. (2015). Human-level control through deep reinforcement learning. *Nature*.
- van Hasselt, Guez & Silver (2016). Deep RL with Double Q-learning. *AAAI-2016*.
- Schaul et al. (2016). Prioritized Experience Replay. *ICLR 2016*.
- Harris (1913). How many parts to make at once. *Factory Magazine*.
- Scarf (1960). The optimality of (s,S) policies. *Mathematical Methods in Social Sciences*.
- Arrow, Harris & Marschak (1951). Optimal inventory policy. *Econometrica*.